# Cross-Firm Momentum via Shared Analyst Coverage
### A hands-on tutorial with hand-computable examples

**Paper:** Mao, Shi, Chen, Wan — *Detecting Cross-Firm Momentum Effects via Shared Analyst Coverage: The Role of Leaders*

**Core idea:** If stock A shares an analyst with stock B, and B is an "information hub" (covered alongside many stocks), then B's price moves first and A's price follows with a delay. We can profit by buying A when B did well last month.

## Step 1: Set up a tiny analyst coverage network

Imagine just **4 stocks** and **3 analysts**:

In [ ]:
import numpy as np
import pandas as pd

# Analyst coverage (which stocks each analyst covers)
analysts = {
    "Analyst_1": {"B", "C", "D"},    # covers 3 stocks
    "Analyst_2": {"A", "B", "D"},    # covers 3 stocks
    "Analyst_3": {"A", "B"},          # covers 2 stocks
}

stocks = ["A", "B", "C", "D"]

# Past month returns (the signal input)
returns = {"A": 0.02, "B": 0.05, "C": -0.01, "D": 0.01}

print("Analyst coverage:")
for a, cov in analysts.items():
    print(f"  {a}: {cov}")
print(f"\nLast month returns: {returns}")

## Step 2: Find connected peers (who shares an analyst with whom?)

In [ ]:
# For each stock, find all other stocks sharing at least one analyst
def find_peers(stock, analysts, all_stocks):
    peers = set()
    for a, covered in analysts.items():
        if stock in covered:
            for s in covered:
                if s != stock:
                    peers.add(s)
    return sorted(peers)

for s in stocks:
    p = find_peers(s, analysts, stocks)
    print(f"  {s} is connected to: {p}")

**Notice:** B is connected to everyone. B is the **hub**. A is only connected to B and D. C is only connected to B and D.

## Step 3: Compute Strength Centrality (SC)

**SC of stock X** = total number of other stocks that share at least one analyst with X.

This measures how far X's analyst coverage "reaches" across the network.

In [ ]:
def compute_sc(stock, analysts, all_stocks):
    sc = 0
    for other in all_stocks:
        if other == stock:
            continue
        # Do stock and other share at least one analyst?
        for a, covered in analysts.items():
            if stock in covered and other in covered:
                sc += 1  # count this connection
                break  # only count once per stock pair
    return sc

sc = {}
for s in stocks:
    sc[s] = compute_sc(s, analysts, stocks)
    print(f"  SC({s}) = {sc[s]}")

print(f"\nB is the hub (SC=3), A/C are peripherals (SC=2)")

Let's verify B's SC by hand:
- B shares Analyst_1 with C and D → 2 connections
- B shares Analyst_2 with A and D → 2 connections (D already counted)
- B shares Analyst_3 with A → 1 connection (A already counted)
- **Unique connections: {A, C, D} → SC(B) = 3** ✓

## Step 4: Compute Common Analysts (n_ij)

For each pair of stocks, count how many analysts cover **both**.

In [ ]:
# n_ij: number of common analysts between stock i and stock j
n_ij = pd.DataFrame(0, index=stocks, columns=stocks)

for a, covered in analysts.items():
    covered_list = sorted(covered)
    for i in range(len(covered_list)):
        for j in range(i + 1, len(covered_list)):
            n_ij.loc[covered_list[i], covered_list[j]] += 1
            n_ij.loc[covered_list[j], covered_list[i]] += 1  # symmetric

print("Common analyst matrix (n_ij):")
print(n_ij)

## Step 5: Compute CF Ret with different weighting schemes

For stock A, the signal is a **weighted average** of its peers' returns.
The weight depends on the scheme.

In [ ]:
def compute_n_j(stock, analysts):
    """Number of analysts covering stock j."""
    return sum(1 for a, cov in analysts.items() if stock in cov)

def compute_cf_ret(focal, analysts, stocks, returns, scheme="SC"):
    peers = find_peers(focal, analysts, stocks)
    if not peers:
        return None
    
    n_j_focal = compute_n_j(focal, analysts)
    weighted_ret = 0
    weights = {}
    
    for p in peers:
        n_ij = n_ij.loc[focal, p]
        n_j = compute_n_j(p, analysts)
        n_j_focal = compute_n_j(focal, analysts)
        
        if scheme == "AH":
            w = n_ij
        elif scheme == "SC":
            w = sc[p]
        elif scheme == "Isr":
            w = n_ij / np.sqrt(max(n_j * n_j_focal, 1))
        elif scheme == "MRX":
            w = n_ij / max(n_j + n_j_focal - n_ij, 1)
        elif scheme == "Sor":
            w = 2 * n_ij / max(n_j + n_j_focal, 1)
        
        weights[p] = w
        weighted_ret += w * returns[p]
    
    # Normalize
    total_w = sum(weights.values())
    if total_w == 0:
        return None
    return weighted_ret / total_w, weights

# --- Compute A's signal under AH (equal common-analyst weighting) ---
cf_ah, w_ah = compute_cf_ret("A", analysts, stocks, returns, "AH")
print("=== Stock A's signal: AH weighting ===")
for p, w in w_ah.items():
    print(f"  Peer {p}: return={returns[p]*100:+.1f}%, weight={w:.3f}, contribution={w*returns[p]*100:+.3f}%")
print(f"  >>> CF Ret^AH(A) = {cf_ah*100:.2f}%")

# --- Compute A's signal under SC (strength centrality weighting) ---
cf_sc, w_sc = compute_cf_ret("A", analysts, stocks, returns, "SC")
print("\n=== Stock A's signal: SC weighting ===")
for p, w in w_sc.items():
    print(f"  Peer {p}: return={returns[p]*100:+.1f}%, SC={sc[p]}, weight={w:.3f}, contribution={w*returns[p]*100:+.3f}%")
print(f"  >>> CF Ret^SC(A) = {cf_sc*100:.2f}%")

**See the difference?**

| Peer | Return | AH weight | SC weight |
|------|--------|-----------|----------|
| B | +5% | 1/3 = 0.33 | 3/5 = 0.60 |
| D | +1% | 2/3 = 0.67 | 2/5 = 0.40 |

Under AH, B and D get roughly equal weight (2 common analysts vs 1).
Under SC, B dominates (60%) because B is the hub. 

Since B had the best return (+5%), the SC signal is **higher** than the AH signal. SC amplifies the leader's signal.

## Step 6: Compute signals for ALL stocks

In [ ]:
results = []
for s in stocks:
    for scheme in ["AH", "SC"]:
        val, _ = compute_cf_ret(s, analysts, stocks, returns, scheme)
        if val is not None:
            results.append({"Stock": s, "Scheme": scheme, "Signal": f"{val*100:.2f}%"})

df = pd.DataFrame(results)
print(df.pivot(index="Stock", columns="Scheme", values="Signal"))

print("""
Trading rule: sort all stocks by signal, buy top decile, short bottom decile.
Here with 4 stocks: buy the highest signal, short the lowest.
""")

## Step 7: Why does this predict returns? (The intuition)

The signal is **purely price-based** — no news, no earnings, no fundamentals. So why does it work?

**The mechanism:**

1. A fundamental shock hits (e.g., industry demand rises)
2. **B** (the hub, SC=3) is on every analyst's screen → price adjusts **fast**
3. **A** (peripheral, SC=2) is on fewer screens → price adjusts **slowly**
4. Our signal captures B's move in month 1 → we buy A → A catches up in month 2 → profit

**Why SC beats AH:** AH treats B and D equally (similar common analyst counts). But B is the information hub — its return is a stronger signal. SC correctly identifies this.

## Step 8: The directional spillover (no reverse)

Let's check: when computing B's signal, does A's return matter?

In [ ]:
cf_b_ah, w_b_ah = compute_cf_ret("B", analysts, stocks, returns, "AH")
cf_b_sc, w_b_sc = compute_cf_ret("B", analysts, stocks, returns, "SC")

print("=== B's signal under AH ===")
for p, w in w_b_ah.items():
    print(f"  Peer {p}: ret={returns[p]*100:+.1f}%, weight={w:.3f}")
print(f"  >>> CF Ret^AH(B) = {cf_b_ah*100:.2f}%")

print("\n=== B's signal under SC ===")
for p, w in w_b_sc.items():
    print(f"  Peer {p}: ret={returns[p]*100:+.1f}%, SC={sc[p]}, weight={w:.3f}")
print(f"  >>> CF Ret^SC(B) = {cf_b_sc*100:.2f}%")

print("""
A's return (+2%) barely affects B's signal under SC (A's weight = 0.20).
B's return (+5%) heavily affects A's signal under SC (B's weight = 0.60).
→ Information flows FROM hubs TO peripherals, not the other way around.
""")

## Summary

| Concept | What it means |
|---------|---------------|
| **Connected peers** | Stocks sharing ≥1 analyst with the focal stock |
| **SC (Strength Centrality)** | How many stocks share an analyst with X (network reach) |
| **CF Ret** | Weighted average return of connected peers |
| **Signal** | High CF Ret → buy (peer leaders did well, focal stock should follow) |
| **Why SC works** | Hubs adjust first, peripherals lag → SC amplifies the leader signal |
| **Directional** | Hub→periphery only (no reverse spillover) |